# 14d · 只有几十个测量的时候

## 这一章在整个项目里的位置

```
00–13 ✅ 平台 → [第二阶段 F]
                 ├─ 14a 稳定性换成 kcal/mol ✅
                 ├─ 14b 从测序读数到适应度 ✅
                 ├─ 14c 一个数拆成两个能量 ✅
                 ├─ 14d 少量数据 + 主动学习      ← 你在这里
                 ├─ 14e 位点地图
                 ├─ 14f 药理伴侣
                 └─ 14g 门面和讲法
```

## 背景：前面所有章节问的问题，公司里不这么问

07–09 和 14a–14c 都在回答同一种问题：**给一个全新的蛋白，一个测量都没有，预测得多准？**

但一个真实的蛋白工程项目不是这样的。它是这样的：

> 我们要改造这个酶。实验室这个月能测 **96 个**突变体。测哪 96 个？测完之后，下个月再测 96 个，又该测哪些？

这一章回答两个很具体的问题：

| 问题 | 为什么重要 |
|---|---|
| **测多少个才值得？** | 测量很贵。如果测 24 个还不如不测（直接用免费的预测），那就别测 |
| **测哪些？** | 同样是 96 个，挑着测能不能比随便测找到更好的突变 |

这是**机器学习引导的定向进化**（machine-learning-guided directed evolution）的核心，也是蛋白工程岗位每天在做的事。

## 怎么做实验

拿我们**锁死的考试蛋白**当"全新的项目"——模型从没见过它们。对每一组实验：

```
假装只测了 n 个突变（n = 24 / 48 / 96 / 384，随机抽，重复 5 次）
         ↓
用这 n 个训练一个小模型
         ↓
在剩下的几千个突变上看排得准不准
```

比的四种做法（前三种是 Hsu 等 2022 年定下的标准对照）：

| 做法 | 用到什么 | 代表什么 |
|---|---|---|
| **零样本** | 什么都不测，直接用 ESM-2 的预测 | 免费的基准线 |
| **one-hot + ridge** | 只知道"第几位换成什么" | 不含任何生物学知识的下限 |
| **ESM-2 向量 + ridge** | 语言模型的残基向量 | 标准做法 |
| **ESM-2 向量 + 零样本分 + ridge** | 再加上零样本预测当一个特征 | Hsu 的"增强"版，公认最稳 |

代码在 `qafi_core.lown`，性质写成了测试。

---

## 准备

In [ ]:
import os, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
from databricks.sdk import WorkspaceClient
from qafi_platform import tracking
from qafi_core import lown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
EXPERIMENT = f"/Users/{w.current_user.me().user_name}/qafi-variant-effect"
MODELS = ROOT / "data/models"

---

## 第 1 步 · 把考试蛋白当成新项目

### 在干什么
取锁死的考试蛋白和它们的 ESM-2 向量，准备好"假装只测了几个"的实验。

### 为什么用考试蛋白
它们满足"全新项目"的全部条件：模型训练时没见过，连它们的亲戚都没见过（07a 按序列相似度整组留出）。而且每组实验有几千个真实测量，可以反复抽样模拟。

### 执行

In [ ]:
test = pd.read_parquet(MODELS / "test_variants.parquet")
index = pd.read_parquet(MODELS / "test_embeddings.parquet")
embeddings = np.asarray(np.load(MODELS / "test_embeddings.npy", mmap_mode="r"))
where = index.assign(row=np.arange(len(index))).set_index(["uniprot_id", "position"]).row
test["row"] = where.reindex(pd.MultiIndex.from_arrays([test.uniprot_id, test.uniprot_position])).values
test = (test.dropna(subset=["row"]).astype({"row": int})
            .rename(columns={"uniprot_position": "position", "alt_aa": "mut_aa"}))
test["predicted"] = test.esm_delta                      # 零样本分：一个测量都不要
big = test.groupby("dataset_id").filter(lambda g: len(g) >= 600)
print(f"{big.dataset_id.nunique()} 组实验、{big.uniprot_id.nunique()} 个考试蛋白、{len(big):,} 个突变")
print(big.groupby("data_kind").dataset_id.nunique().rename("组数").to_string())

### 结果怎么看
128 组实验、92 个蛋白。每组至少 600 个突变，这样抽掉 384 个还能剩下够多的来评判。

---

## 第 2 步 · 测多少个才值得

### 在干什么
对每组实验，随机抽 n 个当"已测"，训练，在剩下的上面看 Spearman。每个预算重复 5 次。

### 背景：为什么零样本是一条水平线
零样本不拟合任何东西——它不看你测了多少个。所以它在图上是一条**水平线**，正好当标尺：**拟合出来的模型，从第几个测量开始才值得？**

### 执行
128 组 × 4 个预算 × 5 次 × 4 个模型，约 10 分钟。

In [ ]:
curves = [lown.curve(assay.reset_index(drop=True), embeddings, draws=5).assign(dataset_id=name)
          for name, assay in big.groupby("dataset_id")]
curve = pd.concat(curves, ignore_index=True)
per_assay = curve.groupby(["dataset_id", "model", "budget"]).spearman.median().reset_index()
table = per_assay.pivot_table(index="model", columns="budget", values="spearman")
table.loc[["zero-shot", "one-hot + ridge", "ESM-2 + ridge", "ESM-2 + zero-shot + ridge"]].round(3)

### 结果怎么看

| 测了几个 | 24 | 48 | 96 | 384 |
|---|---|---|---|---|
| **零样本（什么都不测）** | **0.420** | **0.420** | 0.419 | 0.421 |
| one-hot + ridge | 0.175 | 0.272 | 0.396 | 0.579 |
| ESM-2 + ridge | 0.323 | 0.388 | 0.448 | 0.558 |
| **ESM-2 + 零样本 + ridge** | 0.347 | 0.414 | **0.479** | **0.595** |

三条很实用的结论：

**1. 测 24 个还不如不测。** 零样本 0.420，最好的拟合模型只有 0.347。几十个测量撑不起一个模型，反而被噪声带偏——**这时候该把钱花在别处**。

**2. 大约 96 个是转折点。** 到 96 个时增强模型 0.479 才稳稳超过零样本 0.420；48 个时还打平（0.414）。所以如果目标只是"把突变排个序"，**一块 96 孔板是最低起投金额**。

**3. 没有生物学知识的编码要贵得多。** one-hot 在 24 个时只有 0.175，要到 384 个才追上来（0.579）。语言模型的向量把起步价压低了一大截——**这就是预训练模型在小数据上的价值**。

另外：Hsu 的"增强"技巧（把零样本分当成一个额外特征）在**每一个**预算上都比纯 ESM 向量好。一行代码的事，白拿。

### 面试怎么说
> "I measured the crossover: below about 50 measured variants a zero-shot protein language model ranks better than anything fitted on them, and the fitted model only takes over around one 96-well plate. One-hot encoding needs roughly four times as many measurements to catch up. Adding the zero-shot score as a feature to the ridge — Hsu's augmented model — helps at every budget."

---

## 第 3 步 · 测哪些：主动学习

### 在干什么
模拟一个真实的设计循环：测一批、重新拟合、挑下一批、再测。

### 背景

前一步是**随机**抽 n 个。但真实项目是一轮一轮来的，每一轮可以用已有的结果决定下一轮测什么。这叫**主动学习**（active learning）。

两种挑法：

| 挑法 | 怎么选下一批 | 想法 |
|---|---|---|
| **贪心**（greedy） | 预测最好的那 96 个 | 直接往好的方向挖 |
| **UCB** | 预测值 + 不确定性 | 也去看看模型吃不准的地方，免得钻进死胡同 |

对照组是**随机**挑。这里用三个 ridge（不同正则强度）当一个小集成，它们之间的分歧就是"不确定性"。

### 怎么算赢
不是看相关系数——项目不关心你整体排得准不准，只关心**有没有把最好的那几个挖出来**。所以看两个数：

- **找到的最好值**（占整个景观里最好值的比例）
- **找到了几个前 1%**

### 执行
每轮 96 个，4 轮（共 384 个），三种策略各跑 3 个随机种子。

In [ ]:
campaigns = [lown.campaign(assay.reset_index(drop=True), embeddings, batch=96, rounds=4,
                           strategy=strategy, seed=seed).assign(dataset_id=name)
             for name, assay in big.groupby("dataset_id")
             for strategy in ("random", "greedy", "ucb") for seed in range(3)]
campaign = pd.concat(campaigns, ignore_index=True)
campaign.groupby(["strategy", "round"])[["best found", "top 1% found"]].median().round(3)

### 结果怎么看

| 轮数（每轮 96 个） | 随机：最好值 | 随机：前 1% | 贪心：最好值 | 贪心：前 1% | UCB：最好值 | UCB：前 1% |
|---|---|---|---|---|---|---|
| 1（96 个，都是随机起步） | 0.869 | 1 | 0.869 | 1 | 0.869 | 1 |
| 2（192 个） | 0.905 | 2 | 0.944 | 4 | 0.944 | 4 |
| 3（288 个） | 0.939 | 3 | 0.976 | 5 | 0.976 | 5 |
| 4（384 个） | 0.960 | 4 | 0.998 | 6 | **0.999** | **7** |

第一轮三种策略一样，因为还没有数据可用，都是随机起步。从第二轮开始分化：

- **找到的最好值**：引导挑选基本摸到了全局最优（0.998 / 0.999），随机挑停在 0.960。
- **找到几个前 1%**：UCB 挖出 **7 个**，随机只有 **4 个**——**同样的预算，好东西多了近一倍**。
- UCB 略胜贪心，差距不大。这符合文献：加探索有用，但在这种"位置独立"的景观上有限。

**换成项目语言**：同样是 4 块 96 孔板，让模型挑比随便挑多找到约 75% 的优质突变。

### 面试怎么说
> "I ran the loop a protein engineering project actually runs: 96 per round, four rounds, refit in between. Guided selection reaches 99.8% of the best variant in the landscape against 96.0% for random picking, and finds 7 of the top 1% against 4. The metric is what the project cares about — how many good variants you hold at the end — not the overall correlation."

---

## 第 4 步 · 记账

In [ ]:
best = per_assay.pivot_table(index="model", columns="budget", values="spearman")
last = campaign[campaign["round"] == 4].groupby("strategy")[["best found", "top 1% found"]].median()
metrics = {f"spearman_{int(b)}_{m.replace(' ', '_').replace('+', 'and')}": float(best.loc[m, b])
           for m in best.index for b in best.columns}
metrics |= {f"round4_{s}_{k.replace(' ', '_').replace('%', 'pct')}": float(last.loc[s, k])
            for s in last.index for k in last.columns}
tracking.experiment_run(
    EXPERIMENT, "low_n_and_design",
    params={"assays": int(big.dataset_id.nunique()), "proteins": int(big.uniprot_id.nunique()),
            "budgets": "24,48,96,384", "draws": 5, "batch": 96, "rounds": 4, "seeds": 3,
            "proteins_are": "locked test side"},
    metrics=metrics, tables={"curve": per_assay, "campaign": campaign}, chapter="14d")
print("已记录"); last.round(3)

---

## 这一章做完了什么

- 把锁死的考试蛋白当成 128 个"全新项目"，量出了**测量开始划算的门槛**：约 **96 个**（一块孔板）；低于 50 个时，免费的零样本预测更好
- 量出了**预训练的价值**：one-hot 编码要约 4 倍的测量才能追上语言模型向量
- 跑了一个真实形状的**设计循环**：每轮 96 个、4 轮，引导挑选比随机挑多找到约 75% 的前 1% 突变（7 个对 4 个），并基本摸到全局最优

## 下一步

**14e · 位点地图**：用 14c 的两个能量给全蛋白组的每个位置贴标签（结构关键 / 功能关键 / 可以改），进湖、进 API，并检查"功能敏感但稳定性不敏感"的位置是不是真的落在已知的活性位点和已成药的变构口袋上。

## 提交

In [ ]:
run("git status --short")

In [ ]:
run("git add -A && git commit -q -m 'feat: 14d how many measurements are worth it, and which ones' && git log --oneline -1")